# Educational example 2 — carbon footprint of 1 km in an electric vehicle

A step up from the tea demo: **manufacturing vs use-phase** competition, a value in the **denominator** (`lifetime_km` — the analogue of panel lifetime `LT` in the PV study), and a **discrete design choice drawn from an uncertain probability** (battery recycling — the `bin_*`/`pi_*` pattern from the real case).

`GWI/km = consumption·grid_CO₂ + (glider_CO₂ + capacity·battery_CO₂·(1 − recycle·credit)) / lifetime_km`

> Run with a **numpy < 2** kernel.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import bw2data as bd, brightway2 as bw
from scipy.stats import norm, uniform, triang, lognorm, bernoulli, beta
from SALib.analyze import delta

assert int(np.__version__.split('.')[0]) < 2, 'use a numpy < 2 kernel (bw2data 3.6.x)'

## Layer 1 — build the parameterised project

Four activities: grid electricity, battery (per kWh, with an end-of-life recycling credit baked into its formula), the vehicle (glider + battery pack), and the functional unit `1 km driven`.

In [ ]:
bd.projects.set_current('ev_demo')
bd.Database('demo_bio').write({('demo_bio','co2'):{'name':'carbon dioxide','categories':('air',),'type':'emission','unit':'kg'}})
m = bd.Method(('demo','GWP')); m.register(); m.write([(('demo_bio','co2'), 1.0)])

bd.Database('ev').write({
  ('ev','electricity'): {'name':'grid electricity, 1 kWh','unit':'kWh','exchanges':[
      {'input':('ev','electricity'),'amount':1,'type':'production'},
      {'input':('demo_bio','co2'),'amount':0.4,'type':'biosphere','formula':'grid_CO2'}]},
  ('ev','battery'): {'name':'battery, per kWh','unit':'kWh','exchanges':[
      {'input':('ev','battery'),'amount':1,'type':'production'},
      {'input':('demo_bio','co2'),'amount':80,'type':'biosphere',
       'formula':'battery_CO2_per_kWh * (1 - bin_recycling * recycling_credit)'}]},
  ('ev','vehicle'): {'name':'vehicle','unit':'unit','exchanges':[
      {'input':('ev','vehicle'),'amount':1,'type':'production'},
      {'input':('demo_bio','co2'),'amount':6000,'type':'biosphere','formula':'glider_CO2'},
      {'input':('ev','battery'),'amount':60,'type':'technosphere','formula':'battery_capacity'}]},
  ('ev','km'): {'name':'1 km driven (BEV)','unit':'km','exchanges':[
      {'input':('ev','km'),'amount':1,'type':'production'},
      {'input':('ev','electricity'),'amount':0.18,'type':'technosphere','formula':'consumption'},
      {'input':('ev','vehicle'),'amount':5e-6,'type':'technosphere','formula':'1 / lifetime_km'}]},
})
fu = bd.Database('ev').get('km'); method = bd.Method(('demo','GWP'))

## Layer 2 — parameter registry

Now with a PERT helper and a Bernoulli whose probability is itself uncertain (`pi_recycling` → `bin_recycling`). `pi_recycling` never appears in a formula — it enters the GSA *through* the switch.

In [ ]:
def pert(size, min, mode, max, shape=4):
    a = 1+shape*(mode-min)/(max-min); b = 1+shape*(max-mode)/(max-min)
    return beta.rvs(a, b, size=size)*(max-min)+min
def _bern(size, p):                       # array p -> one Bernoulli draw per element (dependent)
    return bernoulli.rvs(p, size=size) if np.ndim(p)==0 else bernoulli.rvs(p)
SAMPLERS = {
    'normal':     lambda size, loc, scale: norm.rvs(loc, scale, size=size),
    'uniform':    lambda size, low, width: uniform.rvs(low, width, size=size),
    'triangular': lambda size, c, loc, scale: triang.rvs(c, loc, scale, size=size),
    'lognormal':  lambda size, s, scale: lognorm.rvs(s, scale=scale, size=size),
    'pert':       lambda size, **a: pert(size, **a),
    'bernoulli':  _bern,
}
PARAMETERS = [
    dict(name='consumption',         dist='normal',     args=dict(loc=0.18, scale=0.02),         desc='Energy use (kWh/km)'),
    dict(name='grid_CO2',            dist='triangular', args=dict(c=0.43, loc=0.05, scale=0.75), desc='Grid intensity (kg CO2/kWh)'),
    dict(name='lifetime_km',         dist='normal',     args=dict(loc=200000, scale=40000),      desc='Vehicle lifetime (km)'),
    dict(name='battery_capacity',    dist='uniform',    args=dict(low=40, width=40),             desc='Pack size (kWh)'),
    dict(name='battery_CO2_per_kWh', dist='lognormal',  args=dict(s=np.log(1.25), scale=80),     desc='Battery making (kg CO2/kWh)'),
    dict(name='glider_CO2',          dist='lognormal',  args=dict(s=np.log(1.15), scale=6000),   desc='Glider making (kg CO2/veh)'),
    dict(name='recycling_credit',    dist='uniform',    args=dict(low=0.2, width=0.3),           desc='Battery CO2 avoided if recycled'),
    dict(name='pi_recycling',        dist='pert',       args=dict(min=0.3, mode=0.6, max=0.9),   desc='P(battery recycled) [indirect]'),
    dict(name='bin_recycling',       dist='bernoulli',  args=dict(p='pi_recycling'),             desc='Recycling happens? (0/1)'),
]

## Layer 3 — presampling

In [ ]:
np.random.seed(42); N = 5000
sampled = {}
for p in PARAMETERS:
    a = {k:(sampled[v] if isinstance(v,str) else v) for k,v in p['args'].items()}
    sampled[p['name']] = SAMPLERS[p['dist']](size=N, **a)
names = [p['name'] for p in PARAMETERS]
X = np.column_stack([sampled[n] for n in names])
print('X shape:', X.shape)

## Layers 6–7 — the Monte Carlo engine

`run_mc` is identical to the production workflow: for each scenario it writes every `formula` exchange from that row's parameters and runs the LCA. Because **all** parameterised exchanges are overwritten every iteration, the result depends only on the sample matrix `X` — not on leftover database state.

In [ ]:
def run_mc(X, names, fexc, fu, method, n=None):
    n = n or len(X); out = np.empty(n)
    for i in range(n):
        p = dict(zip(names, X[i].tolist()))          # this scenario's parameter values
        for e in fexc:                                # overwrite every formula exchange
            e['amount'] = float(eval(e['formula'], {'__builtins__': None}, p)); e.save()
        lca = bw.LCA({fu: 1}, method.name); lca.lci(); lca.lcia(); out[i] = lca.score
    return out

fexc = [e for a in bd.Database('ev') for e in a.exchanges() if 'formula' in e]
Y = run_mc(X, names, fexc, fu, method)
print(f'mean = {Y.mean():.4g} kg CO2-eq / km   P5 = {np.percentile(Y,5):.4g}   P95 = {np.percentile(Y,95):.4g}')

## Layer 8 — visualise the Monte Carlo output

Histogram + boxplot of the impact distribution (how uncertain the result is).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(Y, bins=40, color='#4c72b0'); ax[0].set_title('Distribution of the impact')
ax[0].set_xlabel('kg CO2-eq / km'); ax[0].set_ylabel('count')
ax[1].boxplot(Y); ax[1].set_xticks([]); ax[1].set_title('Boxplot')
plt.tight_layout(); plt.show()

### Output vs. each input (scatter screening)

One scatter per parameter — the *shape* tells you how it acts: a clear trend = influential, a flat cloud = negligible, two vertical stripes = a binary on/off choice. This is the visual companion to the δ measure.

In [ ]:
ncol = 4; nrow = int(np.ceil(len(names)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3, nrow*2.6), sharey=True)
axes = np.atleast_1d(axes).flatten()
for j, nm in enumerate(names):
    axes[j].scatter(X[:, j], Y, s=5, alpha=0.2, color='#4c72b0')
    axes[j].set_xlabel(nm, fontsize=9)
for k in range(len(names), len(axes)): axes[k].axis('off')
fig.suptitle('Model output vs each input (kg CO2-eq / km)'); plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

## Layer 9 — global sensitivity analysis (Borgonovo δ)

δ ranks how strongly each input drives the output uncertainty (moment-independent, so it handles the binary choices too).

In [ ]:
problem = {'num_vars': len(names), 'names': names, 'bounds': list(zip(X.min(0), X.max(0)))}
df_gsa = delta.analyze(problem, X, Y).to_df().sort_values('delta')
print(df_gsa[['delta','S1']].round(4).iloc[::-1].to_string())

plt.figure(figsize=(7, 0.4*len(names)+1))
plt.barh(range(len(df_gsa)), df_gsa['delta'], color='#c44e52')
plt.yticks(range(len(df_gsa)), df_gsa.index); plt.xlabel('Borgonovo δ')
plt.title('Global sensitivity — which inputs drive the result'); plt.tight_layout(); plt.show()

## What to take away

- Mean ≈ **0.125 kg CO₂-eq/km (125 g/km)**, ~55% use phase / ~45% manufacturing.
- δ ranking: **`grid_CO2` ≫ `lifetime_km` > `consumption` ≈ `battery_capacity` ≈ `battery_CO2_per_kWh` > recycling group**.
- The `bin_recycling` scatter shows two vertical stripes (the 0/1 choice); `lifetime_km` shows a negative/reciprocal trend (it is in the denominator).
- **Three transferable lessons (all mirror the PV study):** (1) grid intensity dominates; (2) a lifetime in the denominator is highly influential — exactly why `LT` topped the PV GSA; (3) the recycling **binary outranks its own probability `pi_recycling`**, because the discrete jump it causes is large. Shrink `grid_CO2`'s range and watch `lifetime_km` take the top spot.